In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB ---- collection 단위로 저장
- Pinecone : 클라우드 vector DB ---- index 단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env에 추가(PINECONE_API_KEY등록)


# 0. 패키지 설치

In [3]:
%pip install -q pinecone langchain-pinecone --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


# 1. knowlege base 구성을 위한 데이터 생성

In [3]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # character 단위로 짜름
loader = Docx2txtLoader('tax_docx/소득세법(법률)(제21221호)(20260701).docx')
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, #문서를 쪼갤 떄 1500글자씩 chunking
        chunk_overlap=20, #200문자씩 overlap 하여 chunking 작업
        #separators=["\n\n","\n"," "] -> 기본값   : 다음의 순서대로 chunking 시도
)
documents = loader.load_and_split(text_splitter=text_splitter)
print('chunk 갯수: ', len(documents))

chunk 갯수:  180


In [6]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding= OpenAIEmbeddings(model='text-embedding-3-large')
len(embedding.embed_query('샘플'))

3072

In [9]:
%%time
#pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
pc=Pinecone(#api_key=os.getenv('PINECONE_API_KEY')
            )
# 데이터를 처음 저장할 때 
# index_name ='tax-index-openai'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드된 벡터 DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name,
)


CPU times: total: 0 ns
Wall time: 1.43 ms


# 2. 답변 생성을 위한 Retrieval

In [10]:
query="연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3)

In [11]:
retrieved_doc="\n--\n".join([doc.page_content.replace("\n\n","") for doc in retrieved_docs])
#print(retrieved_doc)

# 3. 답변생성

In [12]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model='gpt-4.1-nano')

In [32]:
from langchain_upstage import ChatUpstage
llm=ChatUpstage(model='solar-pro2')

In [13]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [14]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉이 5천만원인 직장인의 소득세 금액은 여러 요소에 따라 달라질 수 있지만, 일반적인 경우를 가정하여 설명드리겠습니다.

1. 근로소득과 과세표준 계산
- 총급여액: 50,000,000원
- 근로소득공제: 근로소득공제는 근로소득 금액에 따라 차등 적용되며, 5,000만원의 경우 대략 1,507만원 내지 1,593만원 정도 공제됩니다. 정확한 공제액은 근로소득공제 표에 따라 계산해야 하지만, 대략 1,550만원 정도로 예상할 수 있습니다.
- 과세표준: 50,000,000원 - 약 15,500,000원(근로소득공제) ≈ 약 34,500,000원

2. 세율 적용
- 2023년 기준으로 근로소득에 대한 세율은 누진세 구조로, 과세표준 34,500,000원에 대한 세율은 대략 6%에서 시작하여 최고 15%까지 적용됩니다.
- 예상 세액은 다음과 같습니다.
  - 1,200,000원 이하에 대한 6% 적용
  - 이후 구간별로 누진세율 적용

3. 예상 소득세액
- 대략 계산하면, 약 1,300,000원에서 1,700,000원 정도의 소득세를 부담하실 것으로 예상됩니다.
- 이는 표준공제(연 1,500만원 이하인 경우 인적공제, 보험료공제, 소득공제 등 추가 공제 포함)와 각종 세액공제(본인 공제, 부양가족공제 등)를 감안하지 않은 대략적인 수치임을 참고하시기 바랍니다.

참고 사항:
- 기타 소득이 있거나 세액공제, 특별공제(기부금, 보험료 등)가 적용되는 경우 최종 세액은 달라질 수 있습니다.
- 정확한 세액을 계산하려면 연말정산 또는 근로소득원천세액 계산서 등을 참고하셔야 하며, 국세청 홈택스의 연말정산 계산기 활용도 가능합니다.

요약하면, 연봉 5천만원인 직장인의 예상 소득세는 대략 1,300만 원에서 1,700만 원 사이라고 할 수 있습니다. 더 정확한 계산을 원하시면 구체적인 공제 항목과 공제액, 기타 소득 내역 등을 알려주시거나 세무사와 상담하시는 것이 좋습니다.


# 4. langchain으로 답변 생성

In [15]:
from langchain_core.prompts import ChatPromptTemplate
promptTemplate = ChatPromptTemplate([
                     ('system', '당신은 최고의 한국 소득세법 전문가입니다'),
                     ('human', f'''다음 문맥을 참고하여 질문에 답변하세요. 답을 모르면 모른다고 답변하세요.
    최대 3문장으로 간결하게 답변하세요.
    질문 : {{question}}
    문맥 : {{context}}
    답변 : ''')
                    ]) 

In [16]:
prompt = promptTemplate.invoke({'context':retrieved_doc, 
                                'question':query})

In [17]:
#과금
ai_message = llm.invoke(prompt)

In [18]:
from langchain_core.output_parsers import StrOutputParser
outputParser = StrOutputParser()
outputParser.invoke(ai_message)

'한국의 소득세는 납세자의 과세표준과 세율에 따라 차등 부과됩니다. 연봉 5천만 원인 직장인은 근로소득에 대해 일정 공제 후 세율을 적용받으며, 대략적인 세금액은 약 800만 원~1,500만 원 수준입니다. 정확한 세액은 공제 항목, 기타 소득 또는 공제 대상이 되는 소득공제 등을 고려하여 산출해야 합니다.'

In [19]:
# 위의 작업을 한번에
outputParser.invoke(llm.invoke(promptTemplate.invoke(
    {'context':retrieved_doc,'question':query}
)))

'연봉 5천만원인 직장인이 납부해야 할 소득세는 여러 공제와 세율에 따라 달라지나, 일반적으로 근로소득기본공제와 표준세율을 적용하면 약 1,000만 원 내외의 세액이 예상됩니다. 정확한 세액 계산을 위해서는 공제사항 및 기타 소득 내용을 고려해야 합니다.'

In [20]:
rag_chain =promptTemplate | llm | outputParser
rag_chain.invoke({'context':retrieved_doc, 'question': query})


'연봉 5천만원인 근로자는 기본공제, 특별공제, 인적공제 등을 고려한 후 과세표준에 따라 소득세를 납부하게 됩니다. 2023년 기준으로, 연간 소득세는 대략 1,000만 원 내외로 예상됩니다. 정확한 세액은 공제 사항과 세율에 따라 달라질 수 있으므로 구체적인 계산이 필요합니다.'